In [ ]:
!pip install -q -U huggingface_hub safetensors


In [ ]:
import os
import json
import copy
import math
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torchvision.transforms as transforms

from PIL import Image
from tqdm import tqdm
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torchvision.models import mobilenet_v2
from safetensors.torch import save_file, load_file
from huggingface_hub import HfApi, hf_hub_download

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

if HF_TOKEN is None or HF_TOKEN == "":
    raise ValueError("HF_TOKEN is missing. Add it in Kaggle: Add-ons → Secrets → HF_TOKEN")

api = HfApi(token=HF_TOKEN)
print("[HF] Token loaded.")


In [ ]:
# ============================================================
# CONFIG — MIX EXPERIMENT / FEDAVG DEFAULT
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NUM_CLASSES = 7
IMAGE_SIZE = 224
BATCH_SIZE = 32
LOCAL_EPOCHS = 1
LEARNING_RATE = 1e-4

TRAINING_SCENARIO = "quantityskew"
AGGREGATION_METHOD = "CW-Risk-UCFedAvg"  # "FedAvg", "UCFedAvg", hoặc "CW-Risk-UCFedAvg"

EXPERIMENT_FOLDER = "cw_risk_ucfedavg"
METHOD_NAME = "CW-Risk-UCFedAvg"
LOCAL_TRAINING_METHOD = "FedAvg"  
FEDPROX_MU = 0.01  
ENABLE_CONFORMAL_METRICS = True
CONFORMAL_ALPHA = 0.10
TARGET_COVERAGE = 0.90
UCFEDAVG_BETA = 0.5
UCFEDAVG_LAMBDA_F1 = 0.4
UCFEDAVG_LAMBDA_AUC = 0.25
UCFEDAVG_LAMBDA_COVERAGE = 0.35
UCFEDAVG_LAMBDA_EFFICIENCY = 0.25

# ============================================================
# CW-Risk-UCFedAvg config
# ============================================================
# Class order đúng với label_map trong dataset: nv, mel, bkl, bcc, akiec, vasc, df
CLASS_NAMES = ["nv", "mel", "bkl", "bcc", "akiec", "vasc", "df"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}

# Các class cần chú ý trong HAM10000: hiếm hoặc có rủi ro y khoa.
RARE_CLASS_NAMES = ["mel", "bcc", "akiec", "vasc", "df"]

CLASS_RISK_WEIGHTS = {
    "nv": 1.00,
    "bkl": 1.10,
    "vasc": 1.20,
    "df": 1.20,
    "bcc": 1.40,
    "akiec": 1.50,
    "mel": 1.60,
}


CW_LAMBDA_F1 = 0.30
CW_LAMBDA_RECALL = 0.30
CW_LAMBDA_COVERAGE = 0.25
CW_LAMBDA_EFFICIENCY = 0.15

CW_CLASS_COUNT_BETA = 0.5
CW_RARE_RHO = 0.3
CW_CLASS_COUNT_THRESHOLD = 20
CW_DRIFT_TAU = 

SAMPLE_CONFIDENCE_THRESHOLD = 500
RARE_FACTOR_GAMMA = 0.3



SERVER_REPO = "ChisDong/ServerFL_quantity"


MAX_ROUNDS = 50
EARLY_STOPPING_PATIENCE = 3
MIN_DELTA = 1e-4

KAGGLE_DATA_DIR = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"

IMAGE_DIRS = [
    f"{KAGGLE_DATA_DIR}/HAM10000_images_part_1",
    f"{KAGGLE_DATA_DIR}/HAM10000_images_part_2",
]

SPLIT_DATA_DIR = "/kaggle/input/datasets/chisdong"
SCENARIO_DIR_NAME = "quantityskew"

TEST_CSV = f"{SPLIT_DATA_DIR}/global-test/global_test.csv"

CLIENT_CONFIGS = [
    {
        "client_id": "hospital_AMD",
        "client_repo": "ChisDong/Client_AMD_quantity",
        "train_csv": f"{SPLIT_DATA_DIR}/{SCENARIO_DIR_NAME}/client_AMD_train.csv",
        "val_csv": f"{SPLIT_DATA_DIR}/{SCENARIO_DIR_NAME}/client_AMD_val.csv"
    },
    {
        "client_id": "hospital_NVIDIA",
        "client_repo": "ChisDong/Client_NVIDIA_quantity",
        "scenario": TRAINING_SCENARIO,
        "train_csv": f"{SPLIT_DATA_DIR}/{SCENARIO_DIR_NAME}/client_NVIDIA_train.csv",
        "val_csv": f"{SPLIT_DATA_DIR}/{SCENARIO_DIR_NAME}/client_NVIDIA_val.csv",
    },
    {
        "client_id": "hospital_INTEL",
        "client_repo": "ChisDong/Client_INTEL_quantity",
        "train_csv": f"{SPLIT_DATA_DIR}/{SCENARIO_DIR_NAME}/client_INTEL_train.csv",
        "val_csv": f"{SPLIT_DATA_DIR}/{SCENARIO_DIR_NAME}/client_INTEL_val.csv"
    }
]

CLIENT_REPOS = [client["client_repo"] for client in CLIENT_CONFIGS]
REQUIRED_CLIENTS = len(CLIENT_REPOS)

print("[CONFIG] Device:", DEVICE)
print("[CONFIG] Server repo:", SERVER_REPO)
print("[CONFIG] Client repos:", CLIENT_REPOS)
print("[CONFIG] Required clients:", REQUIRED_CLIENTS)
print("[CONFIG] Max rounds:", MAX_ROUNDS)
print("[CONFIG] Patience:", EARLY_STOPPING_PATIENCE)
print("[CONFIG] Experiment folder:", EXPERIMENT_FOLDER)
print("[CONFIG] Method:", METHOD_NAME)
print("[CONFIG] Local training method:", LOCAL_TRAINING_METHOD)
print("[CONFIG] FedProx mu:", FEDPROX_MU)
print("[CONFIG] Conformal metrics enabled:", ENABLE_CONFORMAL_METRICS)
print("[CONFIG] UCFedAvg beta:", UCFEDAVG_BETA)
print("[CONFIG] Target coverage:", TARGET_COVERAGE)
print("[CONFIG] Rare classes:", RARE_CLASS_NAMES)
print("[CONFIG] Class risk weights:", CLASS_RISK_WEIGHTS)


In [ ]:
# ============================================================
# PATH AUTO-DETECTION & CHECKS
# ============================================================

def find_ham10000_root():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "HAM10000_metadata.csv" in files:
            return root
    return None


def find_file_in_kaggle(filename):
    matches = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if filename in files:
            matches.append(os.path.join(root, filename))
    return matches


# Auto fix HAM10000 path if needed
if not all(os.path.exists(d) for d in IMAGE_DIRS):
    detected_root = find_ham10000_root()
    if detected_root:
        KAGGLE_DATA_DIR = detected_root
        IMAGE_DIRS = [
            f"{KAGGLE_DATA_DIR}/HAM10000_images_part_1",
            f"{KAGGLE_DATA_DIR}/HAM10000_images_part_2",
        ]

print("[PATH] HAM10000 root:", KAGGLE_DATA_DIR)
for image_dir in IMAGE_DIRS:
    print("[PATH] IMAGE_DIR:", image_dir, os.path.exists(image_dir))

# Auto fix TEST_CSV if needed
if not os.path.exists(TEST_CSV):
    matches = find_file_in_kaggle("global_test.csv")
    print("[PATH] global_test.csv matches:", matches)
    if len(matches) > 0:
        TEST_CSV = matches[0]

print("[PATH] TEST_CSV:", TEST_CSV, os.path.exists(TEST_CSV))

# Auto fix client CSV paths if needed
for client in CLIENT_CONFIGS:
    if not os.path.exists(client["train_csv"]):
        matches = find_file_in_kaggle(os.path.basename(client["train_csv"]))
        print(f"[PATH] {os.path.basename(client['train_csv'])} matches:", matches)
        if matches:
            client["train_csv"] = matches[0]
    if not os.path.exists(client["val_csv"]):
        matches = find_file_in_kaggle(os.path.basename(client["val_csv"]))
        print(f"[PATH] {os.path.basename(client['val_csv'])} matches:", matches)
        if matches:
            client["val_csv"] = matches[0]

for client in CLIENT_CONFIGS:
    print("\n[CLIENT PATH]", client["client_id"])
    print("train_csv:", client["train_csv"], os.path.exists(client["train_csv"]))
    print("val_csv:", client["val_csv"], os.path.exists(client["val_csv"]))

if not os.path.exists(TEST_CSV):
    raise FileNotFoundError(f"Missing global_test.csv: {TEST_CSV}")

for image_dir in IMAGE_DIRS:
    if not os.path.exists(image_dir):
        raise FileNotFoundError(f"Missing image folder: {image_dir}")

df_test = pd.read_csv(TEST_CSV)
print("\n[GLOBAL TEST] samples:", len(df_test))
print(df_test["dx"].value_counts())


In [ ]:
# ============================================================
# DATASET
# ============================================================

class HAM10000Dataset(Dataset):
    def __init__(self, csv_file, image_dirs, transform=None):
        self.df = pd.read_csv(csv_file)
        self.image_dirs = image_dirs if isinstance(image_dirs, list) else [image_dirs]
        self.transform = transform

        self.label_map = {
            "nv": 0,
            "mel": 1,
            "bkl": 2,
            "bcc": 3,
            "akiec": 4,
            "vasc": 5,
            "df": 6
        }

        self._validate_columns()
        self.image_path_map = self._build_image_path_map()
        self._validate_images_exist()

        print("[DATASET] CSV:", csv_file)
        print("[DATASET] Samples:", len(self.df))
        print("[DATASET] Indexed images:", len(self.image_path_map))

    def _validate_columns(self):
        required_columns = ["image_id", "dx"]
        missing = set(required_columns) - set(self.df.columns)
        if missing:
            raise ValueError(f"Missing required columns in CSV: {missing}")

    def _build_image_path_map(self):
        image_path_map = {}

        for image_dir in self.image_dirs:
            if not os.path.exists(image_dir):
                print(f"[WARNING] Image dir does not exist: {image_dir}")
                continue

            for file_name in os.listdir(image_dir):
                if file_name.lower().endswith((".jpg", ".jpeg", ".png")):
                    image_id = os.path.splitext(file_name)[0]
                    image_path_map[image_id] = os.path.join(image_dir, file_name)

        return image_path_map

    def _validate_images_exist(self):
        missing = []

        for image_id in self.df["image_id"].astype(str):
            image_id = image_id.strip()
            if image_id not in self.image_path_map:
                missing.append(image_id)

        if missing:
            print("[ERROR] Missing images examples:", missing[:20])
            print("[ERROR] Total missing:", len(missing))
            raise FileNotFoundError(
                f"{len(missing)} images in CSV were not found in IMAGE_DIRS. "
                f"First missing image_id: {missing[0]}"
            )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_id = str(row["image_id"]).strip()
        label_name = str(row["dx"]).strip()

        if label_name not in self.label_map:
            raise ValueError(f"Unknown label: {label_name}")

        image_path = self.image_path_map[image_id]
        image = Image.open(image_path).convert("RGB")
        label = self.label_map[label_name]

        if self.transform:
            image = self.transform(image)

        return image, label


In [ ]:
# ============================================================
# MODEL + METRICS + EARLY STOPPING
# ============================================================


def safe_float(value):
    """
    Convert value to float safely for JSON/metrics.
    NaN/Inf/None -> None.
    """
    if value is None:
        return None

    try:
        value = float(value)
    except (TypeError, ValueError):
        return None

    if math.isnan(value) or math.isinf(value):
        return None

    return value


def sanitize_for_json(obj):
    """
    Convert numpy values and NaN/Inf to JSON-safe values.
    NaN/Inf -> None, which becomes null in JSON.
    """
    if isinstance(obj, dict):
        return {key: sanitize_for_json(value) for key, value in obj.items()}

    if isinstance(obj, list):
        return [sanitize_for_json(value) for value in obj]

    if isinstance(obj, tuple):
        return [sanitize_for_json(value) for value in obj]

    if isinstance(obj, np.ndarray):
        return sanitize_for_json(obj.tolist())

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return safe_float(obj)

    if isinstance(obj, float):
        return safe_float(obj)

    return obj


def to_json_string(obj):
    return json.dumps(
        sanitize_for_json(obj),
        indent=4,
        ensure_ascii=False,
        allow_nan=False
    )


def build_model(num_classes=7):
    model = mobilenet_v2(weights="DEFAULT")
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


def calculate_specificity(y_true, y_pred, num_classes=7):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    specificities = []

    for i in range(num_classes):
        tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        fp = cm[:, i].sum() - cm[i, i]
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(specificity)

    return float(np.mean(specificities))


def calculate_metrics(y_true, y_pred, y_prob=None, num_classes=7):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),

        # SEN = Sensitivity = Recall
        "sensitivity_macro": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),

        "specificity_macro": calculate_specificity(
            y_true, y_pred, num_classes=num_classes
        ),

        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),

        "f1_macro": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),

        "confusion_matrix": confusion_matrix(
            y_true, y_pred, labels=list(range(num_classes))
        ).tolist()
    }

    if y_prob is not None:
        y_prob = np.array(y_prob)
        try:
            metrics["auc_macro_ovr"] = roc_auc_score(
                y_true,
                y_prob,
                multi_class="ovr",
                average="macro",
                labels=list(range(num_classes))
            )
        except ValueError as e:
            print("[AUC ERROR]", e)
            metrics["auc_macro_ovr"] = None
    else:
        metrics["auc_macro_ovr"] = None

    return metrics


def calculate_optimization_score(metrics):
    auc = safe_float(metrics.get("test_auc_macro_ovr")) or 0.0
    sen = safe_float(metrics.get("test_sensitivity_macro")) or 0.0
    f1 = safe_float(metrics.get("test_f1_macro")) or 0.0
    spe = safe_float(metrics.get("test_specificity_macro")) or 0.0

    score = (
        0.40 * auc
        + 0.30 * sen
        + 0.20 * f1
        + 0.10 * spe
    )

    return float(score)


def check_early_stopping(old_metadata, current_score, current_round):
    best_score = old_metadata.get("best_optimization_score", -1)
    best_score = safe_float(best_score)
    if best_score is None:
        best_score = -1.0

    current_score = safe_float(current_score) or 0.0

    patience_counter = old_metadata.get("patience_counter", 0)
    improved = current_score > best_score + MIN_DELTA

    if improved:
        patience_counter = 0
        should_stop = False
        stop_reason = None
    else:
        patience_counter += 1
        should_stop = patience_counter >= EARLY_STOPPING_PATIENCE
        stop_reason = (
            f"No improvement for {EARLY_STOPPING_PATIENCE} consecutive rounds."
            if should_stop else None
        )

    if current_round >= MAX_ROUNDS:
        should_stop = True
        stop_reason = f"Reached MAX_ROUNDS = {MAX_ROUNDS}."

    return {
        "improved": improved,
        "patience_counter": int(patience_counter),
        "should_stop": bool(should_stop),
        "stop_reason": stop_reason
    }


# ============================================================
# CONFORMAL + CLASS-WISE METRICS FOR UCFEDAVG / CW-RISK-UCFEDAVG
# ============================================================

def _safe_div(numerator, denominator, default=0.0):
    denominator = float(denominator)
    if denominator <= 0:
        return default
    return float(numerator) / denominator


def _get_class_name(class_idx):
    try:
        return IDX_TO_CLASS.get(int(class_idx), str(class_idx))
    except NameError:
        default_names = ["nv", "mel", "bkl", "bcc", "akiec", "vasc", "df"]
        return default_names[int(class_idx)] if int(class_idx) < len(default_names) else str(class_idx)


def get_class_counts_from_labels(labels, num_classes=7):
    labels = np.array(labels).astype(int)
    counts = {}
    for c in range(num_classes):
        counts[_get_class_name(c)] = int(np.sum(labels == c))
    return counts


def compute_per_class_classification_metrics(y_true, y_pred, num_classes=7):
    """
    Per-class precision/recall/F1 từ y_true/y_pred.
    Dùng để đánh giá class hiếm và để tính Reliability_{k,c}.
    """
    y_true = np.array(y_true).astype(int)
    y_pred = np.array(y_pred).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    result = {}
    for c in range(num_classes):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        support = cm[c, :].sum()

        precision = _safe_div(tp, tp + fp)
        recall = _safe_div(tp, tp + fn)
        f1 = _safe_div(2 * precision * recall, precision + recall)

        result[_get_class_name(c)] = {
            "support": int(support),
            "precision": safe_float(precision),
            "recall": safe_float(recall),
            "f1": safe_float(f1),
        }

    return result


def compute_conformal_metrics(model, dataloader, device, num_classes=7, alpha=0.10):
    """
    Split-conformal style metrics for client reliability.

    Client-level outputs:
    - coverage
    - avg_prediction_set_size
    - mean_nonconformity
    - q_hat
    - f1_macro
    - auc_macro (optional; None nếu local val thiếu class)

    Class-wise outputs:
    - per_class[class_name].coverage
    - per_class[class_name].avg_prediction_set_size
    - per_class[class_name].mean_nonconformity
    - per_class[class_name].precision / recall / f1
    - per_class[class_name].count
    """
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            if isinstance(batch, (list, tuple)):
                images = batch[0]
                labels = batch[1]
            else:
                raise ValueError("Unsupported dataloader batch format. Expected (images, labels).")

            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            probs = torch.softmax(logits, dim=1)

            all_probs.append(probs.detach().cpu())
            all_labels.append(labels.detach().cpu())

    if len(all_probs) == 0:
        return {
            "conformal_available": False,
            "f1_macro": None,
            "auc_macro": None,
            "coverage": None,
            "avg_prediction_set_size": None,
            "mean_nonconformity": None,
            "q_hat": None,
            "alpha": float(alpha),
            "class_counts": {},
            "rare_class_ratio": None,
            "per_class": {},
        }

    all_probs = torch.cat(all_probs, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy().astype(int)
    y_pred = np.argmax(all_probs, axis=1)

    true_class_probs = all_probs[np.arange(len(all_labels)), all_labels]
    scores = 1.0 - true_class_probs

    q_hat = np.quantile(scores, 1.0 - alpha)
    prediction_sets = (1.0 - all_probs) <= q_hat

    set_sizes = prediction_sets.sum(axis=1)
    contains_true_label = prediction_sets[np.arange(len(all_labels)), all_labels]

    coverage = contains_true_label.mean()
    avg_prediction_set_size = set_sizes.mean()
    mean_nonconformity = scores.mean()

    f1_macro = f1_score(
        all_labels,
        y_pred,
        average="macro",
        zero_division=0
    )

    # AUC local có thể không xác định nếu local validation chỉ có một class.
    try:
        if len(np.unique(all_labels)) < 2:
            auc_macro = None
        else:
            auc_macro = roc_auc_score(
                all_labels,
                all_probs,
                multi_class="ovr",
                average="macro",
                labels=list(range(num_classes))
            )
    except Exception as e:
        print("[CONFORMAL AUC WARNING]", e)
        auc_macro = None

    class_counts = get_class_counts_from_labels(all_labels, num_classes=num_classes)
    rare_names = globals().get("RARE_CLASS_NAMES", ["mel", "bcc", "akiec", "vasc", "df"])
    rare_count = sum(int(class_counts.get(name, 0)) for name in rare_names)
    rare_class_ratio = _safe_div(rare_count, len(all_labels))

    per_class_cls = compute_per_class_classification_metrics(
        y_true=all_labels,
        y_pred=y_pred,
        num_classes=num_classes
    )

    per_class = {}
    for c in range(num_classes):
        class_name = _get_class_name(c)
        mask = all_labels == c
        count = int(mask.sum())

        if count > 0:
            class_coverage = contains_true_label[mask].mean()
            class_avg_set_size = set_sizes[mask].mean()
            class_mean_nonconformity = scores[mask].mean()
            class_efficiency = 1.0 / max(float(class_avg_set_size), 1e-8)
            class_coverage_quality = 1.0 - abs(float(class_coverage) - float(1.0 - alpha))
            class_coverage_quality = max(0.0, min(1.0, class_coverage_quality))
        else:
            class_coverage = None
            class_avg_set_size = None
            class_mean_nonconformity = None
            class_efficiency = None
            class_coverage_quality = None

        per_class[class_name] = {
            "count": count,
            "coverage": safe_float(class_coverage),
            "coverage_quality": safe_float(class_coverage_quality),
            "avg_prediction_set_size": safe_float(class_avg_set_size),
            "conformal_efficiency": safe_float(class_efficiency),
            "mean_nonconformity": safe_float(class_mean_nonconformity),
            "precision": safe_float(per_class_cls[class_name]["precision"]),
            "recall": safe_float(per_class_cls[class_name]["recall"]),
            "f1": safe_float(per_class_cls[class_name]["f1"]),
            "support": int(per_class_cls[class_name]["support"]),
        }

    return {
        "conformal_available": True,
        "f1_macro": safe_float(f1_macro),
        "auc_macro": safe_float(auc_macro),
        "coverage": safe_float(coverage),
        "avg_prediction_set_size": safe_float(avg_prediction_set_size),
        "mean_nonconformity": safe_float(mean_nonconformity),
        "q_hat": safe_float(q_hat),
        "alpha": float(alpha),
        "class_counts": class_counts,
        "rare_class_ratio": safe_float(rare_class_ratio),
        "per_class": per_class,
    }


def build_ucfedavg_metric_view(metadata):
    """
    Chuẩn hóa metadata client thành các key mà UCFedAvg cần.
    AUC chỉ là optional; nếu None thì không ép thành 0 để tránh phạt sai client Non-IID thiếu class.
    """
    return {
        "client_id": metadata.get("client_id"),
        "num_samples": int(metadata.get("samples", metadata.get("train_samples", 0)) or 0),
        "f1_macro": safe_float(metadata.get("conformal_f1_macro"))
            if metadata.get("conformal_f1_macro") is not None else safe_float(metadata.get("val_f1_macro")),
        "auc_macro": safe_float(metadata.get("conformal_auc_macro"))
            if metadata.get("conformal_auc_macro") is not None else safe_float(metadata.get("val_auc_macro_ovr")),
        "coverage": safe_float(metadata.get("conformal_coverage")),
        "avg_prediction_set_size": safe_float(metadata.get("conformal_avg_prediction_set_size")),
        "mean_nonconformity": safe_float(metadata.get("conformal_mean_nonconformity")),
        "rare_class_ratio": safe_float(metadata.get("rare_class_ratio")) or 0.0,
        "class_counts": metadata.get("class_counts", {}) or {},
        "per_class_conformal": metadata.get("per_class_conformal", {}) or {},
    }


In [ ]:
# ============================================================
# HUGGING FACE HELPERS
# ============================================================


def hf_remote_path(path):
    """
    Đưa toàn bộ artifact của FedProx vào cùng một folder trên HF repo.
    Ví dụ: latest/global_model.safetensors -> fedprox/latest/global_model.safetensors
    """
    path = str(path).lstrip("/")

    if path.startswith(f"{EXPERIMENT_FOLDER}/"):
        return path

    return f"{EXPERIMENT_FOLDER}/{path}"


def upload_file_to_hf(local_file, repo_id, remote_file):
    api.upload_file(
        path_or_fileobj=local_file,
        path_in_repo=remote_file,
        repo_id=repo_id,
        repo_type="model"
    )
    print(f"[HF] Uploaded to {repo_id}: {remote_file}")


def download_json_from_hf(repo_id, filename):
    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        token=HF_TOKEN
    )

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def download_safetensors_from_hf(repo_id, filename):
    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        token=HF_TOKEN
    )
    return load_file(path)


def repo_file_exists(repo_id, filename):
    try:
        files = api.list_repo_files(repo_id=repo_id, repo_type="model")
        return filename in files
    except Exception as e:
        print(f"[HF] Cannot list repo files: {repo_id}")
        print(e)
        return False


def save_json(path, obj):
    folder = os.path.dirname(path)

    if folder:
        os.makedirs(folder, exist_ok=True)

    obj = sanitize_for_json(obj)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            obj,
            f,
            indent=4,
            ensure_ascii=False,
            allow_nan=False
        )


In [ ]:
# ============================================================
# SERVER SIDE
# ============================================================

def init_global_model():
    print("[SERVER] Initializing global model round 0...")

    model = build_model(NUM_CLASSES).to(DEVICE)

    global_state_cpu = {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    }

    round_id = 0
    global_model_file = f"global_model_round_{round_id}.safetensors"
    save_file(global_state_cpu, global_model_file)

    server_metadata = {
        "current_round": round_id,
        "latest_global_model": hf_remote_path(global_model_file),
        "latest_global_model_latest_path": hf_remote_path("latest/global_model.safetensors"),

        "best_global_model": hf_remote_path(global_model_file),
        "best_global_model_path": hf_remote_path("best/global_model.safetensors"),
        "best_round": round_id,
        "best_optimization_score": -1,
        "best_metrics": None,

        "patience_counter": 0,

        "status": "ready_for_clients",
        "training_scenario": TRAINING_SCENARIO,
        "aggregation_method": AGGREGATION_METHOD,
        "method": METHOD_NAME,
        "local_training_method": LOCAL_TRAINING_METHOD,
        "fedprox_mu": FEDPROX_MU,
        "experiment_folder": EXPERIMENT_FOLDER,

        "num_classes": NUM_CLASSES,
        "image_size": IMAGE_SIZE,
        "model_architecture": "MobileNetV2",

        "required_clients": REQUIRED_CLIENTS,
        "client_repos": CLIENT_REPOS,

        "early_stopping": {
            "enabled": True,
            "patience": EARLY_STOPPING_PATIENCE,
            "min_delta": MIN_DELTA,
            "max_rounds": MAX_ROUNDS,
            "patience_counter": 0,
            "should_stop": False,
            "stop_reason": None
        },

        "created_at": datetime.utcnow().isoformat(),
        "message": "Initial global model round 0 created."
    }

    metadata_file = "server_metadata.json"
    save_json(metadata_file, server_metadata)

    upload_file_to_hf(global_model_file, SERVER_REPO, hf_remote_path(global_model_file))
    upload_file_to_hf(global_model_file, SERVER_REPO, hf_remote_path("latest/global_model.safetensors"))
    upload_file_to_hf(global_model_file, SERVER_REPO, hf_remote_path("best/global_model.safetensors"))
    upload_file_to_hf(metadata_file, SERVER_REPO, hf_remote_path("server_metadata.json"))

    print("[SERVER] Initial global model uploaded successfully.")
    print(to_json_string(server_metadata))

    return server_metadata


def load_server_metadata():
    metadata = download_json_from_hf(
        repo_id=SERVER_REPO,
        filename=hf_remote_path("server_metadata.json")
    )
    return metadata


def download_current_global_model(server_metadata):
    latest_global_model = server_metadata["latest_global_model"]
    print(f"[SERVER] Downloading current global model: {latest_global_model}")
    return download_safetensors_from_hf(
        repo_id=SERVER_REPO,
        filename=latest_global_model
    )


def download_client_updates(current_round):
    client_updates = []

    for client_repo in CLIENT_REPOS:
        print("\n" + "=" * 80)
        print(f"[SERVER] Checking client repo: {client_repo}")
        print("=" * 80)

        try:
            client_metadata = download_json_from_hf(
                repo_id=client_repo,
                filename=hf_remote_path("latest/metadata.json")
            )

            print("[SERVER] Client metadata:")
            print(to_json_string(client_metadata))

            client_round = int(client_metadata["round"])
            client_status = client_metadata["status"]

            if client_round != current_round:
                print(
                    f"[SERVER] Skip {client_repo}: "
                    f"client round {client_round} != server round {current_round}"
                )
                continue

            if client_status != "completed":
                print(f"[SERVER] Skip {client_repo}: status={client_status}")
                continue

            if "delta_file" not in client_metadata:
                print(f"[SERVER] Skip {client_repo}: missing delta_file")
                continue

            samples = int(client_metadata.get("samples", 0))
            if samples <= 0:
                print(f"[SERVER] Skip {client_repo}: invalid samples={samples}")
                continue

            delta_filename = client_metadata["delta_file"]
            delta_remote_path = hf_remote_path(f"rounds/round_{current_round}/{delta_filename}")

            print(f"[SERVER] Downloading delta: {delta_remote_path}")

            delta_state = download_safetensors_from_hf(
                repo_id=client_repo,
                filename=delta_remote_path
            )

            client_updates.append({
                "client_repo": client_repo,
                "metadata": client_metadata,
                "delta": delta_state,
                "samples": samples
            })

            print(f"[SERVER] Loaded update from {client_repo}")

        except Exception as e:
            print(f"[SERVER] Failed to load update from {client_repo}")
            print(e)

    return client_updates


def aggregate_fedavg(global_state, client_updates):
    if len(client_updates) == 0:
        raise ValueError("No client updates available for aggregation.")

    total_samples = sum(update["samples"] for update in client_updates)
    if total_samples <= 0:
        raise ValueError("Total samples must be > 0.")

    print("[SERVER] Aggregating with FedAvg...")
    print("[SERVER] Total samples:", total_samples)

    aggregated_delta = {}

    for key in global_state.keys():
        global_tensor = global_state[key].detach().cpu()

        if torch.is_floating_point(global_tensor):
            aggregated_delta[key] = torch.zeros_like(global_tensor)
        else:
            aggregated_delta[key] = None

    for update in client_updates:
        weight = update["samples"] / total_samples
        update["aggregation_weight"] = float(weight)
        update["aggregation_method_used"] = "FedAvg"
        delta_state = update["delta"]
        client_id = update["metadata"].get("client_id", update["client_repo"])

        print(
            f"[SERVER] Applying update from {client_id} "
            f"weight={weight:.4f}, samples={update['samples']}"
        )

        for key in global_state.keys():
            global_tensor = global_state[key].detach().cpu()

            if not torch.is_floating_point(global_tensor):
                continue

            if key not in delta_state:
                raise KeyError(f"Missing key in client delta: {key}")

            client_delta = delta_state[key].detach().cpu().float()
            aggregated_delta[key] += weight * client_delta

    new_global_state = {}

    for key in global_state.keys():
        global_tensor = global_state[key].detach().cpu()

        if torch.is_floating_point(global_tensor):
            new_global_state[key] = global_tensor + aggregated_delta[key]
        else:
            # Giữ nguyên tensor Long/Int như BatchNorm num_batches_tracked
            new_global_state[key] = global_tensor

    print("[SERVER] FedAvg aggregation completed.")
    return new_global_state



def _weighted_average_delta(global_state, client_updates, aggregation_weights, method_name):
    """Aggregate all floating tensors with one client-level weight per client."""
    aggregated_delta = {}

    for key in global_state.keys():
        global_tensor = global_state[key].detach().cpu()
        if torch.is_floating_point(global_tensor):
            aggregated_delta[key] = torch.zeros_like(global_tensor)
        else:
            aggregated_delta[key] = None

    for update, weight in zip(client_updates, aggregation_weights):
        delta_state = update["delta"]
        client_id = update["metadata"].get("client_id", update["client_repo"])

        print(
            f"[SERVER] Applying {method_name} update from {client_id} "
            f"weight={weight:.4f}, samples={update['samples']}"
        )

        for key in global_state.keys():
            global_tensor = global_state[key].detach().cpu()
            if not torch.is_floating_point(global_tensor):
                continue
            if key not in delta_state:
                raise KeyError(f"Missing key in client delta: {key}")
            client_delta = delta_state[key].detach().cpu().float()
            aggregated_delta[key] += weight * client_delta

    new_global_state = {}
    for key in global_state.keys():
        global_tensor = global_state[key].detach().cpu()
        if torch.is_floating_point(global_tensor):
            new_global_state[key] = global_tensor + aggregated_delta[key]
        else:
            new_global_state[key] = global_tensor

    return new_global_state


def compute_client_reliability(metric_view, target_coverage=TARGET_COVERAGE, eps=1e-8):
    """
    Reliability client-level dùng cho UCFedAvg/backbone.
    Không ép AUC None thành 0. Phiên bản hiện tại không dùng AUC local để tránh lỗi Non-IID thiếu class.
    """
    f1 = safe_float(metric_view.get("f1_macro")) or 0.0
    coverage = safe_float(metric_view.get("coverage"))
    avg_set_size = safe_float(metric_view.get("avg_prediction_set_size"))

    if coverage is None:
        coverage_quality = 0.0
    else:
        coverage_quality = 1.0 - abs(float(coverage) - float(target_coverage))
        coverage_quality = max(0.0, min(1.0, coverage_quality))

    if avg_set_size is None or avg_set_size <= 0:
        conformal_efficiency = 1.0 / float(NUM_CLASSES)
    else:
        conformal_efficiency = 1.0 / max(float(avg_set_size), eps)

    reliability = (
        UCFEDAVG_LAMBDA_F1 * f1
        + UCFEDAVG_LAMBDA_COVERAGE * coverage_quality
        + UCFEDAVG_LAMBDA_EFFICIENCY * conformal_efficiency
    )

    return max(0.0, float(reliability)), {
        "f1_macro": safe_float(f1),
        "coverage": safe_float(coverage),
        "coverage_quality": safe_float(coverage_quality),
        "avg_prediction_set_size": safe_float(avg_set_size),
        "conformal_efficiency": safe_float(conformal_efficiency),
        "reliability": safe_float(reliability),
    }


def compute_ucfedavg_client_weights(
    client_updates,
    beta=UCFEDAVG_BETA,
    target_coverage=TARGET_COVERAGE,
    eps=1e-8,
    method_name="UCFedAvg"
):
    """
    Client-level UCFedAvg weights.

    raw_weight_k = (n_k / N)^beta * reliability_k * sample_confidence_k * rare_factor_k
    """
    if len(client_updates) == 0:
        raise ValueError("No client updates available for UCFedAvg.")

    total_samples = sum(int(update.get("samples", 0)) for update in client_updates)
    if total_samples <= 0:
        raise ValueError("Total samples must be > 0.")

    raw_weights = []
    debug_rows = []

    for update in client_updates:
        metadata = update["metadata"]
        metric_view = build_ucfedavg_metric_view(metadata)

        samples = int(metric_view["num_samples"])
        data_weight = samples / max(total_samples, eps)
        reliability, reliability_debug = compute_client_reliability(metric_view, target_coverage=target_coverage, eps=eps)

        rare_class_ratio = safe_float(metric_view.get("rare_class_ratio")) or 0.0
        rare_factor = 1.0 + RARE_FACTOR_GAMMA * rare_class_ratio
        sample_confidence = min(1.0, samples / max(float(SAMPLE_CONFIDENCE_THRESHOLD), eps))

        raw_weight = (data_weight ** beta) * reliability * sample_confidence * rare_factor
        raw_weights.append(raw_weight)

        debug_row = {
            "client_id": metadata.get("client_id"),
            "samples": samples,
            "data_weight": safe_float(data_weight),
            "sample_confidence": safe_float(sample_confidence),
            "rare_class_ratio": safe_float(rare_class_ratio),
            "rare_factor": safe_float(rare_factor),
            "raw_weight": safe_float(raw_weight),
            **reliability_debug,
        }
        debug_rows.append(debug_row)

    total_raw = sum(raw_weights)

    if total_raw <= eps:
        print(f"[SERVER][{method_name}] total_raw <= eps, fallback to FedAvg weights.")
        aggregation_weights = [float(update.get("samples", 0)) / max(total_samples, eps) for update in client_updates]
    else:
        aggregation_weights = [float(w / total_raw) for w in raw_weights]

    for update, agg_weight, row in zip(client_updates, aggregation_weights, debug_rows):
        update["aggregation_weight"] = float(agg_weight)
        update["aggregation_method_used"] = method_name
        update["ucfedavg_debug"] = row
        update["ucfedavg_reliability"] = row.get("reliability")
        update["ucfedavg_raw_weight"] = row.get("raw_weight")

    print(f"[SERVER][{method_name}] Client-level weighting debug:")
    print(to_json_string(debug_rows))
    print(f"[SERVER][{method_name}] Normalized client-level weights:")
    print(to_json_string([
        {
            "client_id": update["metadata"].get("client_id"),
            "aggregation_weight": update.get("aggregation_weight")
        }
        for update in client_updates
    ]))

    return aggregation_weights


def aggregate_ucfedavg(global_state, client_updates):
    if len(client_updates) == 0:
        raise ValueError("No client updates available for aggregation.")

    print("[SERVER] Aggregating with UCFedAvg...")

    aggregation_weights = compute_ucfedavg_client_weights(
        client_updates=client_updates,
        beta=UCFEDAVG_BETA,
        target_coverage=TARGET_COVERAGE,
        method_name="UCFedAvg"
    )

    new_global_state = _weighted_average_delta(
        global_state=global_state,
        client_updates=client_updates,
        aggregation_weights=aggregation_weights,
        method_name="UCFedAvg"
    )

    print("[SERVER] UCFedAvg aggregation completed.")
    return new_global_state


# ============================================================
# CW-RISK-UCFEDAVG: BACKBONE CLIENT-LEVEL + CLASSIFIER HEAD CLASS-WISE
# ============================================================

def _is_classifier_head_key(key, tensor):
    """
    MobileNetV2 classifier keys thường là:
    - classifier.1.weight: [NUM_CLASSES, hidden_dim]
    - classifier.1.bias: [NUM_CLASSES]
    """
    if not torch.is_floating_point(tensor):
        return False
    if "classifier" not in key:
        return False
    if tensor.ndim >= 1 and tensor.shape[0] == NUM_CLASSES:
        return True
    return False


def _get_class_metric(metadata, class_name):
    per_class = metadata.get("per_class_conformal", {}) or {}
    return per_class.get(class_name, {}) or {}


def compute_class_reliability(metadata, class_name, target_coverage=TARGET_COVERAGE, eps=1e-8):
    """
    Reliability_{k,c} cho classifier-head class-wise aggregation.
    Ưu tiên recall vì bài y tế cần giảm false negative ở class nguy hiểm.
    """
    m = _get_class_metric(metadata, class_name)

    f1 = safe_float(m.get("f1")) or 0.0
    recall = safe_float(m.get("recall")) or 0.0
    coverage = safe_float(m.get("coverage"))
    avg_set_size = safe_float(m.get("avg_prediction_set_size"))

    if coverage is None:
        coverage_quality = 0.0
    else:
        coverage_quality = 1.0 - abs(float(coverage) - float(target_coverage))
        coverage_quality = max(0.0, min(1.0, coverage_quality))

    if avg_set_size is None or avg_set_size <= 0:
        efficiency = 0.0
    else:
        efficiency = 1.0 / max(float(avg_set_size), eps)

    reliability = (
        CW_LAMBDA_F1 * f1
        + CW_LAMBDA_RECALL * recall
        + CW_LAMBDA_COVERAGE * coverage_quality
        + CW_LAMBDA_EFFICIENCY * efficiency
    )

    return max(0.0, float(reliability)), {
        "f1": safe_float(f1),
        "recall": safe_float(recall),
        "coverage": safe_float(coverage),
        "coverage_quality": safe_float(coverage_quality),
        "avg_prediction_set_size": safe_float(avg_set_size),
        "efficiency": safe_float(efficiency),
        "reliability": safe_float(reliability),
    }


def compute_classwise_head_weights(client_updates, eps=1e-8):
    """
    Tính alpha_{k,c} riêng cho từng class c.

    RawHeadWeight_{k,c} =
        (ClassCount_{k,c}+eps)^beta
        * Reliability_{k,c}
        * SampleConfidence_{k,c}
        * RareBoost_c
        * DriftPenalty_{k,c}
    """
    class_names = globals().get("CLASS_NAMES", ["nv", "mel", "bkl", "bcc", "akiec", "vasc", "df"])

    # Tổng count mỗi class trong các client tham gia round này.
    global_class_counts = {name: 0 for name in class_names}
    for update in client_updates:
        counts = update["metadata"].get("class_counts", {}) or {}
        for name in class_names:
            global_class_counts[name] += int(counts.get(name, 0) or 0)

    total_class_count = sum(global_class_counts.values())
    if total_class_count <= 0:
        total_class_count = sum(int(update.get("samples", 0)) for update in client_updates)

    head_weights_by_class = {}
    debug_by_class = {}

    for class_idx, class_name in enumerate(class_names):
        raw_weights = []
        rows = []

        class_total = max(float(global_class_counts.get(class_name, 0)), eps)
        inverse_frequency = max(float(total_class_count), eps) / (len(class_names) * class_total)
        risk_weight = float(CLASS_RISK_WEIGHTS.get(class_name, 1.0))
        rare_boost = risk_weight * (inverse_frequency ** CW_RARE_RHO)

        for update in client_updates:
            metadata = update["metadata"]
            client_id = metadata.get("client_id")
            counts = metadata.get("class_counts", {}) or {}
            class_count = int(counts.get(class_name, 0) or 0)

            reliability, rel_debug = compute_class_reliability(metadata, class_name)
            sample_confidence = min(1.0, class_count / max(float(CW_CLASS_COUNT_THRESHOLD), eps))

            # Drift penalty optional, tính trên classifier row nếu có delta.
            drift_penalty = 1.0
            if CW_DRIFT_TAU and CW_DRIFT_TAU > 0:
                try:
                    drift_norm = 0.0
                    for key, delta_tensor in update["delta"].items():
                        if key in update["delta"] and torch.is_floating_point(delta_tensor):
                            dt = delta_tensor.detach().cpu().float()
                            if "classifier" in key and dt.ndim >= 1 and dt.shape[0] == NUM_CLASSES:
                                row_delta = dt[class_idx]
                                drift_norm += float(torch.norm(row_delta).item())
                    drift_penalty = math.exp(-float(CW_DRIFT_TAU) * drift_norm)
                except Exception:
                    drift_penalty = 1.0

            raw = ((class_count + eps) ** CW_CLASS_COUNT_BETA) * reliability * sample_confidence * rare_boost * drift_penalty
            raw_weights.append(raw)

            rows.append({
                "client_id": client_id,
                "class_name": class_name,
                "class_count": class_count,
                "sample_confidence": safe_float(sample_confidence),
                "risk_weight": safe_float(risk_weight),
                "inverse_frequency": safe_float(inverse_frequency),
                "rare_boost": safe_float(rare_boost),
                "drift_penalty": safe_float(drift_penalty),
                "raw_weight": safe_float(raw),
                **rel_debug,
            })

        total_raw = sum(raw_weights)
        if total_raw <= eps:
            # Fallback: nếu class này không có metric/count, dùng client-level weight đã tính cho backbone.
            normalized = [float(update.get("aggregation_weight", 1.0 / len(client_updates))) for update in client_updates]
            norm_sum = sum(normalized)
            normalized = [w / max(norm_sum, eps) for w in normalized]
        else:
            normalized = [float(w / total_raw) for w in raw_weights]

        for row, w in zip(rows, normalized):
            row["normalized_head_weight"] = safe_float(w)

        head_weights_by_class[class_name] = normalized
        debug_by_class[class_name] = rows

    print("[SERVER][CW-Risk-UCFedAvg] Class-wise head weighting debug:")
    print(to_json_string(debug_by_class))

    return head_weights_by_class, debug_by_class


def aggregate_cw_risk_ucfedavg(global_state, client_updates):
    if len(client_updates) == 0:
        raise ValueError("No client updates available for aggregation.")

    print("[SERVER] Aggregating with CW-Risk-UCFedAvg...")

    # 1) Client-level weights cho backbone / non-classifier tensors.
    backbone_weights = compute_ucfedavg_client_weights(
        client_updates=client_updates,
        beta=UCFEDAVG_BETA,
        target_coverage=TARGET_COVERAGE,
        method_name="CW-Risk-UCFedAvg"
    )

    # 2) Class-wise weights cho classifier head.
    head_weights_by_class, classwise_debug = compute_classwise_head_weights(client_updates)

    new_global_state = {}
    class_names = globals().get("CLASS_NAMES", ["nv", "mel", "bkl", "bcc", "akiec", "vasc", "df"])

    for key in global_state.keys():
        global_tensor = global_state[key].detach().cpu()

        if not torch.is_floating_point(global_tensor):
            new_global_state[key] = global_tensor
            continue

        # Classifier head: aggregate từng row class riêng.
        if _is_classifier_head_key(key, global_tensor):
            aggregated_delta = torch.zeros_like(global_tensor).float()

            for class_idx, class_name in enumerate(class_names):
                weights_c = head_weights_by_class.get(class_name, backbone_weights)

                for update, weight_c in zip(client_updates, weights_c):
                    delta_state = update["delta"]
                    if key not in delta_state:
                        raise KeyError(f"Missing key in client delta: {key}")
                    client_delta = delta_state[key].detach().cpu().float()
                    aggregated_delta[class_idx] += float(weight_c) * client_delta[class_idx]

            new_global_state[key] = global_tensor + aggregated_delta
            print(f"[SERVER][CW-Risk-UCFedAvg] Class-wise aggregated head tensor: {key}")

        # Backbone / feature extractor: aggregate bằng client-level UCFedAvg weight.
        else:
            aggregated_delta = torch.zeros_like(global_tensor).float()
            for update, weight in zip(client_updates, backbone_weights):
                delta_state = update["delta"]
                if key not in delta_state:
                    raise KeyError(f"Missing key in client delta: {key}")
                client_delta = delta_state[key].detach().cpu().float()
                aggregated_delta += float(weight) * client_delta

            new_global_state[key] = global_tensor + aggregated_delta

    # Lưu debug vào từng update để metadata server có thể phân tích sau.
    for update in client_updates:
        update["cw_risk_ucfedavg_classwise_weights"] = {}
        cid = update["metadata"].get("client_id")
        for class_name, rows in classwise_debug.items():
            for row in rows:
                if row.get("client_id") == cid:
                    update["cw_risk_ucfedavg_classwise_weights"][class_name] = row
                    break

    print("[SERVER] CW-Risk-UCFedAvg aggregation completed.")
    return new_global_state

def evaluate_global_model_on_test(global_state, round_id):
    print(f"[SERVER] Evaluating global model round {round_id}...")

    test_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    test_dataset = HAM10000Dataset(
        csv_file=TEST_CSV,
        image_dirs=IMAGE_DIRS,
        transform=test_transform
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    model = build_model(NUM_CLASSES)
    model.load_state_dict(global_state, strict=True)
    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    test_loss, y_true, y_pred, y_prob = evaluate_model(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=DEVICE
    )

    test_metrics = calculate_metrics(
        y_true=y_true,
        y_pred=y_pred,
        y_prob=y_prob,
        num_classes=NUM_CLASSES
    )

    result = {
        "round": round_id,
        "test_samples": len(test_dataset),

        "test_loss": float(test_loss),
        "test_accuracy": float(test_metrics["accuracy"]),
        "test_sensitivity_macro": float(test_metrics["sensitivity_macro"]),
        "test_specificity_macro": float(test_metrics["specificity_macro"]),
        "test_precision_macro": float(test_metrics["precision_macro"]),
        "test_f1_macro": float(test_metrics["f1_macro"]),
        "test_auc_macro_ovr": safe_float(test_metrics["auc_macro_ovr"]),
        "confusion_matrix": test_metrics["confusion_matrix"]
    }

    result["optimization_score"] = calculate_optimization_score(result)

    print("[SERVER] Global test metrics:")
    print(to_json_string(result))

    return result


def publish_new_global_model(new_global_state, old_round, client_updates, old_metadata):
    new_round = old_round + 1
    global_model_file = f"global_model_round_{new_round}.safetensors"

    new_global_state_cpu = {
        key: value.detach().cpu()
        for key, value in new_global_state.items()
    }

    save_file(new_global_state_cpu, global_model_file)
    print(f"[SERVER] Saved new global model: {global_model_file}")

    global_test_metrics = evaluate_global_model_on_test(
        global_state=new_global_state_cpu,
        round_id=new_round
    )

    optimization_score = safe_float(global_test_metrics["optimization_score"]) or 0.0

    metrics_file = f"server_metrics_round_{new_round}.json"
    save_json(metrics_file, global_test_metrics)

    early_stop_info = check_early_stopping(
        old_metadata=old_metadata,
        current_score=optimization_score,
        current_round=new_round
    )

    is_best_model = early_stop_info["improved"]

    previous_best_score = old_metadata.get("best_optimization_score", -1)
    if previous_best_score is None:
        previous_best_score = -1

    if is_best_model:
        best_global_model = hf_remote_path(global_model_file)
        best_round = new_round
        best_optimization_score = optimization_score
        best_metrics = global_test_metrics
        print("[SERVER] New best model found.")
    else:
        best_global_model = old_metadata.get("best_global_model")
        best_round = old_metadata.get("best_round")
        best_optimization_score = previous_best_score
        best_metrics = old_metadata.get("best_metrics")
        print("[SERVER] Best model unchanged.")

    aggregation_summary = []

    for update in client_updates:
        metadata = update["metadata"]

        aggregation_summary.append({
            "client_repo": update["client_repo"],
            "client_id": metadata.get("client_id"),
            "scenario": metadata.get("scenario"),
            "round": metadata.get("round"),

            "samples": metadata.get("samples"),
            "train_samples": metadata.get("train_samples"),
            "val_samples": metadata.get("val_samples"),

            "train_loss": metadata.get("train_loss"),
            "train_accuracy": metadata.get("train_accuracy"),
            "train_sensitivity_macro": metadata.get("train_sensitivity_macro"),
            "train_specificity_macro": metadata.get("train_specificity_macro"),
            "train_precision_macro": metadata.get("train_precision_macro"),
            "train_f1_macro": metadata.get("train_f1_macro"),
            "train_auc_macro_ovr": metadata.get("train_auc_macro_ovr"),

            "val_loss": metadata.get("val_loss"),
            "val_accuracy": metadata.get("val_accuracy"),
            "val_sensitivity_macro": metadata.get("val_sensitivity_macro"),
            "val_specificity_macro": metadata.get("val_specificity_macro"),
            "val_precision_macro": metadata.get("val_precision_macro"),
            "val_f1_macro": metadata.get("val_f1_macro"),
            "val_auc_macro_ovr": metadata.get("val_auc_macro_ovr"),

            "delta_file": metadata.get("delta_file"),

            "aggregation_method_used": update.get("aggregation_method_used"),
            "aggregation_weight": update.get("aggregation_weight"),
            "ucfedavg_reliability": update.get("ucfedavg_reliability"),
            "ucfedavg_raw_weight": update.get("ucfedavg_raw_weight"),
            "ucfedavg_debug": update.get("ucfedavg_debug"),
            "cw_risk_ucfedavg_classwise_weights": update.get("cw_risk_ucfedavg_classwise_weights"),
            "class_counts": metadata.get("class_counts"),
            "rare_class_ratio": metadata.get("rare_class_ratio"),
            "per_class_conformal": metadata.get("per_class_conformal"),

            "conformal_available": metadata.get("conformal_available"),
            "conformal_alpha": metadata.get("conformal_alpha"),
            "conformal_q_hat": metadata.get("conformal_q_hat"),
            "conformal_coverage": metadata.get("conformal_coverage"),
            "conformal_avg_prediction_set_size": metadata.get("conformal_avg_prediction_set_size"),
            "conformal_mean_nonconformity": metadata.get("conformal_mean_nonconformity"),
            "conformal_f1_macro": metadata.get("conformal_f1_macro"),
            "conformal_auc_macro": metadata.get("conformal_auc_macro"),

            "base_global_model": metadata.get("base_global_model")
        })

    status = "stop_training" if early_stop_info["should_stop"] else "ready_for_clients"

    server_metadata = {
        "current_round": new_round,
        "latest_global_model": hf_remote_path(global_model_file),
        "latest_global_model_latest_path": hf_remote_path("latest/global_model.safetensors"),

        "best_global_model": best_global_model,
        "best_global_model_path": hf_remote_path("best/global_model.safetensors"),
        "best_round": best_round,
        "best_optimization_score": best_optimization_score,
        "best_metrics": best_metrics,

        "patience_counter": early_stop_info["patience_counter"],

        "status": status,
        "training_scenario": TRAINING_SCENARIO,
        "aggregation_method": AGGREGATION_METHOD,
        "method": METHOD_NAME,
        "local_training_method": LOCAL_TRAINING_METHOD,
        "fedprox_mu": FEDPROX_MU,
        "experiment_folder": EXPERIMENT_FOLDER,

        "num_classes": NUM_CLASSES,
        "image_size": IMAGE_SIZE,
        "model_architecture": "MobileNetV2",

        "required_clients": REQUIRED_CLIENTS,
        "participated_clients": len(client_updates),
        "client_repos": CLIENT_REPOS,

        "early_stopping": {
            "enabled": True,
            "patience": EARLY_STOPPING_PATIENCE,
            "min_delta": MIN_DELTA,
            "max_rounds": MAX_ROUNDS,
            "patience_counter": early_stop_info["patience_counter"],
            "should_stop": early_stop_info["should_stop"],
            "stop_reason": early_stop_info["stop_reason"]
        },

        "global_test_metrics": global_test_metrics,
        "aggregation_summary": aggregation_summary,

        "aggregated_at": datetime.utcnow().isoformat()
    }

    metadata_file = "server_metadata.json"
    save_json(metadata_file, server_metadata)

    upload_file_to_hf(global_model_file, SERVER_REPO, hf_remote_path(global_model_file))
    upload_file_to_hf(global_model_file, SERVER_REPO, hf_remote_path("latest/global_model.safetensors"))

    if is_best_model:
        upload_file_to_hf(global_model_file, SERVER_REPO, hf_remote_path("best/global_model.safetensors"))

    upload_file_to_hf(metadata_file, SERVER_REPO, hf_remote_path("server_metadata.json"))
    upload_file_to_hf(metrics_file, SERVER_REPO, hf_remote_path(f"metrics/{metrics_file}"))
    upload_file_to_hf(metrics_file, SERVER_REPO, hf_remote_path("latest/server_metrics.json"))

    print(f"[SERVER] Published global model round {new_round}")
    print(to_json_string(server_metadata))

    return server_metadata


def run_aggregation_round():
    print("\n" + "=" * 80)
    print("[SERVER] Starting aggregation round")
    print("=" * 80)

    server_metadata = load_server_metadata()

    current_round = int(server_metadata["current_round"])
    status = server_metadata.get("status", "unknown")

    print(f"[SERVER] Current round: {current_round}")
    print(f"[SERVER] Server status: {status}")

    if status == "stop_training":
        print("[SERVER] Training already stopped.")
        print("[SERVER] Best model:", server_metadata.get("best_global_model"))
        print("[SERVER] Best round:", server_metadata.get("best_round"))
        print("[SERVER] Best score:", server_metadata.get("best_optimization_score"))
        print("[SERVER] Reason:", server_metadata.get("early_stopping", {}).get("stop_reason"))
        return server_metadata

    if status != "ready_for_clients":
        raise RuntimeError(f"[SERVER] Server is not ready. Current status: {status}")

    client_updates = download_client_updates(current_round=current_round)

    print(f"\n[SERVER] Valid client updates: {len(client_updates)}")

    if len(client_updates) < REQUIRED_CLIENTS:
        raise RuntimeError(
            f"[SERVER] Not enough clients. "
            f"Required: {REQUIRED_CLIENTS}, Available: {len(client_updates)}"
        )

    global_state = download_current_global_model(server_metadata)

    aggregation_method = str(AGGREGATION_METHOD).lower()

    if aggregation_method == "fedavg":
        new_global_state = aggregate_fedavg(
            global_state=global_state,
            client_updates=client_updates
        )
    elif aggregation_method == "ucfedavg":
        new_global_state = aggregate_ucfedavg(
            global_state=global_state,
            client_updates=client_updates
        )
    elif aggregation_method in ["cw-risk-ucfedavg", "cw_risk_ucfedavg", "cwucfedavg", "cw-risk-ucfed"]:
        new_global_state = aggregate_cw_risk_ucfedavg(
            global_state=global_state,
            client_updates=client_updates
        )
    else:
        raise ValueError(f"Unsupported AGGREGATION_METHOD: {AGGREGATION_METHOD}")

    new_metadata = publish_new_global_model(
        new_global_state=new_global_state,
        old_round=current_round,
        client_updates=client_updates,
        old_metadata=server_metadata
    )

    print("[SERVER] Aggregation completed successfully.")
    return new_metadata


In [ ]:
# ============================================================
# CLIENT SIDE
# ============================================================

def download_server_metadata(repo_id):
    return download_json_from_hf(repo_id=repo_id, filename=hf_remote_path("server_metadata.json"))


def download_global_model(repo_id):
    metadata = download_server_metadata(repo_id)

    if metadata.get("status") == "stop_training":
        print("[CLIENT] Server requested stop_training.")
        return None, metadata

    if "latest_global_model" not in metadata:
        raise KeyError("latest_global_model missing in server_metadata.json")

    latest_model = metadata["latest_global_model"]

    print(f"[CLIENT] Server round: {metadata['current_round']}")
    print(f"[CLIENT] Downloading global model: {latest_model}")

    state_dict = download_safetensors_from_hf(repo_id=repo_id, filename=latest_model)

    return state_dict, metadata


def train_local(model, loader, criterion, optimizer, device, epochs):
    model.train()

    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0.0
    total_batches = 0

    for epoch in range(epochs):
        progress = tqdm(loader, desc=f"Epoch {epoch + 1}/{epochs}")

        for images, labels in progress:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)

            total_loss += loss.item()
            total_batches += 1

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

            progress.set_postfix(loss=float(loss.item()))

    avg_loss = total_loss / max(total_batches, 1)

    return model, avg_loss, all_labels, all_preds, all_probs


def train_local_fedprox(model, global_model, loader, criterion, optimizer, device, epochs, mu=FEDPROX_MU):
    model.train()
    global_model.eval()

    global_params = {
        name: param.detach().clone().to(device)
        for name, param in global_model.named_parameters()
    }

    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0.0
    total_batches = 0

    for epoch in range(epochs):
        progress = tqdm(loader, desc=f"FedProx Epoch {epoch + 1}/{epochs}")

        for images, labels in progress:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            ce_loss = criterion(outputs, labels)

            proximal_term = 0.0
            for name, param in model.named_parameters():
                proximal_term += torch.sum((param - global_params[name]) ** 2)

            loss = ce_loss + (mu / 2.0) * proximal_term

            loss.backward()
            optimizer.step()

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)

            total_loss += loss.item()
            total_batches += 1

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

            progress.set_postfix(
                ce_loss=float(ce_loss.item()),
                loss=float(loss.item())
            )

    avg_loss = total_loss / max(total_batches, 1)

    return model, avg_loss, all_labels, all_preds, all_probs


def evaluate_model(model, loader, criterion, device):
    model.eval()

    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)

            total_loss += loss.item()
            total_batches += 1

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

    avg_loss = total_loss / max(total_batches, 1)

    return avg_loss, all_labels, all_preds, all_probs


def create_delta(global_state, local_state):
    delta = {}

    for key in global_state.keys():
        if key not in local_state:
            raise KeyError(f"Missing key in local_state: {key}")

        global_tensor = global_state[key].detach().cpu()
        local_tensor = local_state[key].detach().cpu()

        if global_tensor.shape != local_tensor.shape:
            raise ValueError(
                f"Shape mismatch at {key}: "
                f"global={global_tensor.shape}, local={local_tensor.shape}"
            )

        # Chỉ tạo delta cho tensor float, bỏ qua num_batches_tracked dạng Long.
        if torch.is_floating_point(global_tensor):
            delta[key] = local_tensor.float() - global_tensor.float()

    return delta


def run_client_training(client_config):
    os.makedirs("checkpoints", exist_ok=True)
    os.makedirs("metadata", exist_ok=True)

    client_id = client_config["client_id"]
    client_repo = client_config["client_repo"]
    train_csv = client_config["train_csv"]
    val_csv = client_config["val_csv"]
    scenario = client_config.get("scenario", TRAINING_SCENARIO)

    print("\n" + "=" * 80)
    print(f"[CLIENT] Starting training: {client_id}")
    print("=" * 80)

    global_state, server_metadata = download_global_model(SERVER_REPO)

    if server_metadata.get("status") == "stop_training":
        return {
            "client_id": client_id,
            "status": "stop_training",
            "server_metadata": server_metadata
        }

    current_round = int(server_metadata["current_round"])

    train_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    val_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    train_dataset = HAM10000Dataset(
        csv_file=train_csv,
        image_dirs=IMAGE_DIRS,
        transform=train_transform
    )

    val_dataset = HAM10000Dataset(
        csv_file=val_csv,
        image_dirs=IMAGE_DIRS,
        transform=val_transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    model = build_model(NUM_CLASSES)
    model.load_state_dict(global_state, strict=True)
    model = model.to(DEVICE)

    global_model_ref = build_model(NUM_CLASSES)
    global_model_ref.load_state_dict(global_state, strict=True)
    global_model_ref = global_model_ref.to(DEVICE)

    class_weights = torch.tensor([
        1.0,  # nv
        4.0,  # mel
        3.0,  # bkl
        4.0,  # bcc
        5.0,  # akiec
        6.0,  # vasc
        6.0   # df
    ], dtype=torch.float32).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

    if str(LOCAL_TRAINING_METHOD).lower() == "fedprox":
        model, train_loss, train_y_true, train_y_pred, train_y_prob = train_local_fedprox(
            model=model,
            global_model=global_model_ref,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=DEVICE,
            epochs=LOCAL_EPOCHS,
            mu=FEDPROX_MU
        )
    elif str(LOCAL_TRAINING_METHOD).lower() == "fedavg":
        model, train_loss, train_y_true, train_y_pred, train_y_prob = train_local(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=DEVICE,
            epochs=LOCAL_EPOCHS
        )
    else:
        raise ValueError(f"Unsupported LOCAL_TRAINING_METHOD: {LOCAL_TRAINING_METHOD}")

    train_metrics = calculate_metrics(
        y_true=train_y_true,
        y_pred=train_y_pred,
        y_prob=train_y_prob,
        num_classes=NUM_CLASSES
    )

    val_loss, val_y_true, val_y_pred, val_y_prob = evaluate_model(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE
    )

    val_metrics = calculate_metrics(
        y_true=val_y_true,
        y_pred=val_y_pred,
        y_prob=val_y_prob,
        num_classes=NUM_CLASSES
    )

    if ENABLE_CONFORMAL_METRICS:
        conformal_metrics = compute_conformal_metrics(
            model=model,
            dataloader=val_loader,
            device=DEVICE,
            num_classes=NUM_CLASSES,
            alpha=CONFORMAL_ALPHA
        )
    else:
        conformal_metrics = {
            "conformal_available": False,
            "f1_macro": None,
            "auc_macro": None,
            "coverage": None,
            "avg_prediction_set_size": None,
            "mean_nonconformity": None,
            "q_hat": None,
            "alpha": CONFORMAL_ALPHA,
            "class_counts": get_class_counts_from_labels([label for _, label in val_dataset], num_classes=NUM_CLASSES) if False else {},
            "rare_class_ratio": None,
            "per_class": {}
        }

    print("[CLIENT] Conformal metrics:")
    print(to_json_string(conformal_metrics))

    # Class distribution của train set dùng cho rare-class/risk-aware aggregation.
    train_labels_for_counts = train_dataset.df["dx"].map(train_dataset.label_map).astype(int).values
    train_class_counts = get_class_counts_from_labels(train_labels_for_counts, num_classes=NUM_CLASSES)
    rare_train_count = sum(int(train_class_counts.get(name, 0)) for name in RARE_CLASS_NAMES)
    train_rare_class_ratio = rare_train_count / max(len(train_dataset), 1)

    local_state = model.state_dict()
    delta = create_delta(global_state=global_state, local_state=local_state)

    delta_filename = f"{client_id}_round_{current_round}_delta.safetensors"
    metadata_filename = f"{client_id}_round_{current_round}_metadata.json"

    delta_path = f"checkpoints/{delta_filename}"
    metadata_path = f"metadata/{metadata_filename}"

    save_file(delta, delta_path)

    client_metadata = {
        "client_id": client_id,
        "client_repo": client_repo,
        "scenario": scenario,
        "round": current_round,
        "status": "completed",
        "method": METHOD_NAME,
        "local_training_method": LOCAL_TRAINING_METHOD,
        "aggregation_method": AGGREGATION_METHOD,
        "fedprox_mu": FEDPROX_MU,
        "experiment_folder": EXPERIMENT_FOLDER,

        "train_samples": len(train_dataset),
        "val_samples": len(val_dataset),
        "samples": len(train_dataset),
        "class_counts": train_class_counts,
        "rare_class_ratio": safe_float(train_rare_class_ratio),
        "rare_classes": RARE_CLASS_NAMES,
        "per_class_conformal": conformal_metrics.get("per_class", {}),

        "local_epochs": LOCAL_EPOCHS,
        "base_global_model": server_metadata["latest_global_model"],
        "created_at": datetime.utcnow().isoformat(),

        "train_loss": float(train_loss),
        "train_accuracy": float(train_metrics["accuracy"]),
        "train_sensitivity_macro": float(train_metrics["sensitivity_macro"]),
        "train_specificity_macro": float(train_metrics["specificity_macro"]),
        "train_precision_macro": float(train_metrics["precision_macro"]),
        "train_f1_macro": float(train_metrics["f1_macro"]),
        "train_auc_macro_ovr": safe_float(train_metrics["auc_macro_ovr"]),

        "val_loss": float(val_loss),
        "val_accuracy": float(val_metrics["accuracy"]),
        "val_sensitivity_macro": float(val_metrics["sensitivity_macro"]),
        "val_specificity_macro": float(val_metrics["specificity_macro"]),
        "val_precision_macro": float(val_metrics["precision_macro"]),
        "val_f1_macro": float(val_metrics["f1_macro"]),
        "val_auc_macro_ovr": safe_float(val_metrics["auc_macro_ovr"]),

        # Conformal metrics dùng cho UCFedAvg. Với AGGREGATION_METHOD="FedAvg", các field này chỉ để log/phân tích.
        "conformal_available": conformal_metrics.get("conformal_available"),
        "conformal_alpha": safe_float(conformal_metrics.get("alpha")),
        "conformal_q_hat": safe_float(conformal_metrics.get("q_hat")),
        "conformal_coverage": safe_float(conformal_metrics.get("coverage")),
        "conformal_avg_prediction_set_size": safe_float(conformal_metrics.get("avg_prediction_set_size")),
        "conformal_mean_nonconformity": safe_float(conformal_metrics.get("mean_nonconformity")),
        "conformal_f1_macro": safe_float(conformal_metrics.get("f1_macro")),
        "conformal_auc_macro": safe_float(conformal_metrics.get("auc_macro")),

        "delta_file": delta_filename
    }

    save_json(metadata_path, client_metadata)

    # Upload theo round
    upload_file_to_hf(
        local_file=delta_path,
        repo_id=client_repo,
        remote_file=hf_remote_path(f"rounds/round_{current_round}/{delta_filename}")
    )

    upload_file_to_hf(
        local_file=metadata_path,
        repo_id=client_repo,
        remote_file=hf_remote_path(f"rounds/round_{current_round}/{metadata_filename}")
    )

    # Upload latest để server dễ đọc
    upload_file_to_hf(
        local_file=delta_path,
        repo_id=client_repo,
        remote_file=hf_remote_path("latest/delta.safetensors")
    )

    upload_file_to_hf(
        local_file=metadata_path,
        repo_id=client_repo,
        remote_file=hf_remote_path("latest/metadata.json")
    )

    print("[CLIENT] Completed:", client_id)
    print(to_json_string(client_metadata))

    return client_metadata


In [ ]:
# ============================================================
# AUTO FL RUNNER
# ============================================================

def run_auto_fl_training():
    print("\n" + "=" * 80)
    print("[AUTO-FL] Starting automatic FL training with Hugging Face synchronization")
    print("=" * 80)

    for loop_idx in range(MAX_ROUNDS):
        print("\n" + "#" * 80)
        print(f"[AUTO-FL] LOOP {loop_idx + 1}/{MAX_ROUNDS}")
        print("#" * 80)

        server_metadata = load_server_metadata()
        server_status = server_metadata.get("status", "unknown")
        current_round = int(server_metadata["current_round"])

        print(f"[AUTO-FL] Server current_round: {current_round}")
        print(f"[AUTO-FL] Server status: {server_status}")

        if server_status == "stop_training":
            print("[AUTO-FL] Server requested stop_training.")
            print("[AUTO-FL] Best model:", server_metadata.get("best_global_model"))
            print("[AUTO-FL] Best round:", server_metadata.get("best_round"))
            print("[AUTO-FL] Best score:", server_metadata.get("best_optimization_score"))
            break

        # ==============================
        # CLIENTS TRAINING
        # ==============================

        for client_config in CLIENT_CONFIGS:
            client_metadata = run_client_training(client_config)

            if client_metadata.get("status") == "stop_training":
                print("[AUTO-FL] Client detected stop_training.")
                return client_metadata

        # ==============================
        # SERVER AGGREGATION
        # ==============================

        server_metadata = run_aggregation_round()

        print("[AUTO-FL] Aggregated round:", server_metadata.get("current_round"))
        print("[AUTO-FL] Latest model:", server_metadata.get("latest_global_model"))
        print("[AUTO-FL] Best model:", server_metadata.get("best_global_model"))
        print("[AUTO-FL] Best round:", server_metadata.get("best_round"))
        print("[AUTO-FL] Best score:", server_metadata.get("best_optimization_score"))

        early_stopping = server_metadata.get("early_stopping", {})
        if early_stopping.get("should_stop") is True:
            print("\n[AUTO-FL] Early stopping triggered.")
            print("[AUTO-FL] Reason:", early_stopping.get("stop_reason"))
            break

    print("\n" + "=" * 80)
    print("[AUTO-FL] Training finished")
    print("=" * 80)

    final_metadata = load_server_metadata()
    return final_metadata


In [ ]:
final_metadata = run_auto_fl_training()


In [ ]:
# init_metadata = init_global_model()
